# 01 · Sampling, blur & resolution
**Medical Imaging Interactive Labs · 30–45 minutes**

This exercise will explore the image-formation chain:

**Object → optical blur → pixel-area averaging → sampling**

This should help you distinguish blur from coarse sampling, predict an alias frequency, and use an MTF curve to explain loss of contrast. We use the sinusoids, convolution, and sampling theorem discussed in lecture to describe an imaging system.

### Start here
1. Choose **Runtime → Run all** in Colab and wait until the explorer appears. In JupyterLab, choose **Run → Run All Cells**.
2. Scroll to **Resolution explorer**. Release a slider to update. Try **Fine sampling**, **Blur limited**, and **Aliasing**.
3. Complete the four investigations below. The final investigation about phase at the Nyquist frequency is an optional extension.

All lengths are in **mm in the image plane**. We assume the image and object have the same size. Intensities are scaled from 0 to 1, and every image uses the same grayscale limits so that changes in brightness and contrast remain visible.


In [ ]:
#@title Setup · run automatically with Run all
import importlib.util
import subprocess
import sys

# Install only missing packages; avoid replacing Colab's working scientific stack.
required = {"numpy": "numpy>=1.26,<3", "scipy": "scipy>=1.11,<2",
            "matplotlib": "matplotlib>=3.8,<4", "ipywidgets": "ipywidgets>=8.1,<9"}
missing = [spec for name, spec in required.items() if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
try:
    from google.colab import output as colab_output
except ImportError:
    pass  # Standard Jupyter uses its built-in widget manager.
else:
    colab_output.enable_custom_widget_manager()

import numpy as np
import scipy
from scipy.optimize import brentq
import matplotlib
from IPython import get_ipython
get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

print("Ready · Python", sys.version.split()[0], "| NumPy", np.__version__,
      "| SciPy", scipy.__version__, "| Matplotlib", matplotlib.__version__,
      "| ipywidgets", widgets.__version__)


## 1 · The physical model
### The object and its contrast
The resolution target, or object we are imaging, is a pattern of vertical stripes. Its intensity varies sinusoidally along $x$ and is constant along $y$:

$$
I_0(x)=\tfrac12[1+C_0\cos(2\pi f x+\phi)],\qquad C_0=0.9.
$$

The **spatial frequency** $f$ is measured in **cycles/mm**. One bright–dark pair is one cycle, and the stripe period is $1/f$ mm. Increasing $f$ puts the stripes closer together. The phase $\phi$ shifts the stripes along $x$ without changing their spacing.

**Contrast** describes how strongly bright and dark regions differ. For a sinusoidal pattern,

$$
C=\frac{I_{\max}-I_{\min}}{I_{\max}+I_{\min}}.
$$

The object has contrast $C_0=0.9$. Blurring reduces the sinusoid's amplitude while leaving its mean intensity unchanged, so it also reduces contrast.

### Optical blur and the point-spread function
A very small bright point in the object generally produces a spread-out spot in the image. The **point-spread function (PSF)** describes that spot. For a linear, shift-invariant system (LSI in Prince and Links), the image is the convolution of the object with the PSF.

We approximate the PSF with a Gaussian:

$$
h(x,y)=\frac{1}{2\pi\sigma^2}\exp\!\left[-\frac{x^2+y^2}{2\sigma^2}\right].
$$

The parameter $\sigma$, in mm, sets the spot's width. A larger $\sigma$ means more blur. The PSF integrates to one, so blurring a uniform image does not change its intensity. The setting $\sigma=0$ represents an ideal delta-function PSF with no optical blur.

Another measure of the spot's width is its **full width at half maximum (FWHM)**: the distance across the central profile where the PSF is at half its peak value. For a Gaussian,

$$
\mathrm{FWHM}=2\sqrt{2\ln 2}\,\sigma\approx2.355\sigma.
$$

This Gaussian is a useful approximation for studying blur. A real camera's PSF can have a different shape.

### Pixel spacing and averaging
Each detector pixel records the average image intensity over a small square area. The distance between neighboring pixel centers is the **pixel spacing**, also called the **pitch**, $p$. Here, the pixel width equals that spacing: the light-sensitive squares fill the detector without gaps. This finite collecting area is also called the **pixel aperture**.

Larger pixels average over more of the image, which can reduce the contrast of closely spaced stripes. Sampling then assigns one recorded value to each pixel center. Averaging and sampling are separate steps.

The centers are at $x_n=(n+\tfrac12)p$. Because the stripes are constant along $y$, the two-dimensional pixel average reduces to

$$
d_n=\frac{1}{p}\int_{x_n-p/2}^{x_n+p/2} I_{\rm blur}(x)\,dx.
$$

### Modulation transfer function
The **modulation transfer function (MTF)** describes how much sinusoidal contrast an imaging system preserves at each spatial frequency:

$$
\mathrm{MTF}(f)=\frac{C_{\rm out}(f)}{C_{\rm in}(f)}.
$$

For example, an MTF of 0.5 means the output sinusoid has half the input contrast. High spatial frequencies describe fine detail, so the MTF curve shows how well the system transfers that detail.

For our Gaussian PSF, the optical response is

$$
H_{\rm opt}(f)=\exp(-2\pi^2\sigma^2f^2).
$$

Averaging over a pixel contributes $\mathrm{sinc}(pf)$, where $\mathrm{sinc}(u)=\sin(\pi u)/(\pi u)$ and $\mathrm{sinc}(0)=1$. Combining these two effects gives the MTF **before sampling**, also called the **presampling MTF**:

$$
\mathrm{MTF}_{\rm pre}(f)=|H_{\rm opt}(f)\,\mathrm{sinc}(pf)|.
$$

The predicted contrast after blur and pixel averaging is $C_0\,\mathrm{MTF}_{\rm pre}(f)$. **MTF50** is the first frequency where this MTF falls to 0.5. It helps compare systems, but it is not a sharp boundary between visible and invisible detail.

### Sampling and aliasing
The spatial sampling frequency and Nyquist frequency are

$$
f_s=\frac{1}{p},\qquad f_N=\frac{1}{2p}.
$$

Both have units of cycles/mm; $f_s$ also specifies the number of samples per mm. A sinusoid below $f_N$ has more than two samples per period. Above $f_N$, the samples can resemble a lower-frequency pattern: **aliasing**. A clear pattern on the detector therefore does not always represent the object's true stripe spacing.

For the **Aliasing** preset, the object frequency lies between $f_N$ and $f_s$, so its apparent frequency is $f_s-f$. The optional extension examines what happens exactly at $f_N$.


In [ ]:
#@title Image-formation model · exact sinusoidal solution
FOV_MM = 8.0
INPUT_CONTRAST = 0.9
PITCHES_MM = (0.05, 0.10, 0.20, 0.25, 0.40, 0.50)

def optical_otf(f, sigma):
    return np.exp(-2 * np.pi**2 * sigma**2 * np.asarray(f)**2)

def presampling_otf(f, sigma, pitch):
    return optical_otf(f, sigma) * np.sinc(pitch * np.asarray(f))

def alias_frequency(f, pitch):
    fs = 1 / pitch
    return abs((f + fs / 2) % fs - fs / 2)

def sample_detector(f, sigma, pitch, phase_deg):
    n = int(round(FOV_MM / pitch))
    centers = (np.arange(n) + 0.5) * pitch
    response = presampling_otf(f, sigma, pitch)
    samples = 0.5 * (1 + INPUT_CONTRAST * response *
                     np.cos(2 * np.pi * f * centers + np.deg2rad(phase_deg)))
    return centers, samples

def fit_sampled_sinusoid(centers, samples, frequency):
    # At DC/Nyquist the sin/cos basis cannot identify both amplitude and phase.
    design = np.column_stack([np.ones_like(centers),
                              np.cos(2 * np.pi * frequency * centers),
                              np.sin(2 * np.pi * frequency * centers)])
    if np.linalg.cond(design) > 1e10:
        return None
    coefficients = np.linalg.lstsq(design, samples, rcond=None)[0]
    contrast = np.hypot(*coefficients[1:]) / coefficients[0]
    return contrast, coefficients

def mtf50(sigma, pitch):
    # Unique crossing on the first lobe (before the aperture's first zero).
    return brentq(lambda f: presampling_otf(f, sigma, pitch) - 0.5, 0, 1 / pitch)

def measurements(f, sigma, pitch, phase_deg):
    x, values = sample_detector(f, sigma, pitch, phase_deg)
    fa = alias_frequency(f, pitch)
    fit = fit_sampled_sinusoid(x, values, fa)
    return dict(x=x, values=values, alias=fa, nyquist=1/(2*pitch),
                pre_mtf=abs(presampling_otf(f, sigma, pitch)),
                mtf50=mtf50(sigma, pitch), fit=fit,
                sample_range_contrast=np.ptp(values)/(values.max()+values.min()))


## 2 · Resolution explorer
**Read left to right:** object intensity, image after optical blur, and the recorded detector image. Each image shows an 8 mm square. The lower-left plot shows intensity along the first 4 mm of the image; the dots are the detector's pixel averages.

- **Fine sampling:** little optical blur and many pixels per stripe period.
- **Blur limited:** the same pixel spacing, with a wider PSF.
- **Aliasing:** compare the stripe spacing in the object and the detector image.
- **Nyquist boundary:** an optional extension using the phase control.

The MTF plot separates optical blur from pixel averaging and shows their combined effect. The vertical Nyquist line marks the sampling limit for the selected pixel spacing. The PSF plot shows how the blur width compares with the pixel width; both profiles are scaled to a peak of one for this comparison.

The measurement table gives stripe period, pixels per period, PSF FWHM, Nyquist frequency, apparent frequency, and contrast transfer. **Apparent frequency** describes a sinusoid consistent with the samples; it does not establish the original object's frequency. Additional measurements for the extension are available under **Additional measurements**.


In [ ]:
#@title Launch resolution explorer · use the controls below
COLORS = dict(optical="#197c81", aperture="#b57921", total="#35356e", alias="#bd3c53")

def make_dashboard(f=1.0, sigma=0.04, pitch=0.10, phase_deg=0.0):
    m = measurements(f, sigma, pitch, phase_deg)
    phi = np.deg2rad(phase_deg)
    fine_x = (np.arange(800) + 0.5) * FOV_MM / 800
    target = 0.5 * (1 + INPUT_CONTRAST * np.cos(2*np.pi*f*fine_x + phi))
    blurred = 0.5 * (1 + INPUT_CONTRAST * optical_otf(f, sigma) *
                     np.cos(2*np.pi*f*fine_x + phi))
    with plt.rc_context({"font.size": 10, "axes.spines.top": False,
                         "axes.spines.right": False, "figure.facecolor": "white"}):
        fig, axes = plt.subplots(2, 3, figsize=(14, 8.4), constrained_layout=True)
        fig.suptitle(f"Resolution explorer  |  f = {f:g} cycles/mm  ·  σ = {sigma:g} mm"
                     f"  ·  p = {pitch:g} mm  ·  phase = {phase_deg:g}°", fontsize=14)
        images = [np.broadcast_to(target, (400, len(target))),
                  np.broadcast_to(blurred, (400, len(blurred))),
                  np.broadcast_to(m["values"], (len(m["values"]), len(m["values"])))]
        titles = ["1 · Object: sinusoidal target", "2 · Optical image: Gaussian blur",
                  f"3 · Detector: {len(m['x'])} × {len(m['x'])} pixels"]
        for ax, image, title in zip(axes[0], images, titles):
            im = ax.imshow(image, cmap="gray", vmin=0, vmax=1, origin="lower",
                           extent=[0, FOV_MM, 0, FOV_MM], interpolation="nearest")
            ax.set(title=title, xlabel="x (mm)", ylabel="y (mm)", xticks=[0, 2, 4, 6, 8])
        fig.colorbar(im, ax=list(axes[0]), shrink=0.75, label="Normalized intensity")

        ax = axes[1, 0]
        ax.plot(fine_x, blurred, color=COLORS["optical"], label="Optical intensity", lw=1.7)
        ax.plot(m["x"], m["values"], "o", ms=3.5, color=COLORS["alias"], label="Pixel averages")
        if m["fit"] is not None:
            _, c = m["fit"]
            folded = c[0] + c[1]*np.cos(2*np.pi*m["alias"]*fine_x) + c[2]*np.sin(2*np.pi*m["alias"]*fine_x)
            ax.plot(fine_x, folded, "--", lw=1.4, color=COLORS["total"], label="Fit at folded frequency")
        ax.set(title="4 · Profile: first 4 mm", xlabel="x (mm)", ylabel="Normalized intensity",
               xlim=(0, 4), ylim=(-0.03, 1.03))
        ax.legend(fontsize=8, loc="upper right")
        ax.grid(alpha=0.2)

        ax = axes[1, 1]
        # Fixed frequency axis: moving pitch does not disguise changes of bandwidth.
        freq = np.linspace(0, 12, 1601)
        ax.plot(freq, optical_otf(freq, sigma), color=COLORS["optical"], label="Optical MTF")
        ax.plot(freq, abs(np.sinc(pitch*freq)), color=COLORS["aperture"], label="Pixel-averaging MTF")
        ax.plot(freq, abs(presampling_otf(freq, sigma, pitch)), color=COLORS["total"], lw=2, label="Combined MTF before sampling")
        ax.axvspan(m["nyquist"], 12, color=COLORS["alias"], alpha=0.07)
        ax.axvline(m["nyquist"], color=COLORS["alias"], ls=":", label="Nyquist")
        ax.axvline(f, color="0.4", lw=1, ls="--", label="Target frequency")
        ax.scatter([f], [m["pre_mtf"]], color=COLORS["total"], s=35, zorder=5)
        ax.set(title="5 · Contrast transfer before sampling", xlabel="Spatial frequency (cycles/mm)",
               ylabel="MTF (unitless)", xlim=(0, 12), ylim=(-0.02, 1.04))
        ax.legend(fontsize=7.5, loc="upper right")
        ax.grid(alpha=0.2)

        ax = axes[1, 2]
        psf_x = np.linspace(-1.5, 1.5, 1601)
        if sigma > 0:
            # Normalized central cross-section, not the 2-D PSF's absolute units.
            ax.plot(psf_x, np.exp(-0.5*(psf_x/sigma)**2), color=COLORS["optical"], lw=2)
            half_width = np.sqrt(2*np.log(2))*sigma
            ax.hlines(0.5, -half_width, half_width, color=COLORS["alias"], lw=2, label="Optical FWHM")
        else:
            ax.annotate("Ideal delta PSF", xy=(0, 0.98), xytext=(0.25, 0.7),
                        arrowprops={"arrowstyle":"->"})
            ax.vlines(0, 0, 1, color=COLORS["optical"], lw=2)
        ax.plot([-pitch/2, -pitch/2, pitch/2, pitch/2], [0, 1, 1, 0],
                color=COLORS["aperture"], ls="--", label="Pixel width p")
        ax.set(title="6 · PSF width and pixel width", xlabel="Offset x (mm)",
               ylabel="Peak-normalized cross-section", xlim=(-1.5, 1.5), ylim=(0, 1.1))
        ax.legend(fontsize=8, loc="upper right")
        ax.grid(alpha=0.2)
    return fig

def metric_html(f, sigma, pitch, phase_deg):
    m = measurements(f, sigma, pitch, phase_deg)
    if np.isclose(f, m["nyquist"], atol=1e-10, rtol=0):
        status = "NYQUIST BOUNDARY · recorded contrast depends on phase"
    elif f > m["nyquist"]:
        status = "ABOVE NYQUIST · a surviving signal folds to a lower frequency"
    else:
        status = "BELOW NYQUIST · target frequency is within the sampling band"
    fit_text = (f"{m['fit'][0]:.4f}" if m["fit"] is not None else
                "indeterminate (DC or Nyquist; amplitude/phase cannot be separated)")
    sign_text = "yes (negative aperture response)" if presampling_otf(f, sigma, pitch) < -1e-12 else "no"
    rows = [
        ("Target period", f"{1/f:.3f} mm"),
        ("Pixels per target period", f"{1/(pitch*f):.3f}"),
        ("PSF full width at half maximum", f"{2*np.sqrt(2*np.log(2))*sigma:.3f} mm"),
        ("Nyquist frequency", f"{m['nyquist']:.3f} cycles/mm"),
        ("Apparent frequency", f"{m['alias']:.3f} cycles/mm"),
        ("MTF before sampling, at target f", f"{m['pre_mtf']:.4f}"),
        ("Predicted contrast after pixel averaging", f"{INPUT_CONTRAST*m['pre_mtf']:.4f}"),
        ("MTF50 before sampling", f"{m['mtf50']:.3f} cycles/mm"),
]
    table = "".join(f"<tr><td style='padding:3px 18px 3px 0'>{k}</td><td>{v}</td></tr>" for k,v in rows)
    extra_rows = [
        ("Sampled sinusoid fit contrast", fit_text),
        ("Sample range contrast", f"{m['sample_range_contrast']:.4f}"),
        ("Phase reversal from pixel averaging", sign_text)]
    extra = "".join(f"<tr><td style='padding:3px 18px 3px 0'>{k}</td><td>{v}</td></tr>" for k,v in extra_rows)
    return f"<div style='border-left:4px solid #197c81;padding:10px 16px'><b>{status}</b><table>{table}</table><small>Contrast and MTF are unitless. MTF50 can exceed Nyquist; sampling still limits which frequencies can be recorded faithfully.</small><details><summary>Additional measurements</summary><table>{extra}</table></details></div>"

# Close a previous explorer when this cell is rerun.
if "explorer" in globals():
    explorer.close()
style = {"description_width": "170px"}
layout = widgets.Layout(width="min(100%, 650px)")
controls = dict(
    f=widgets.FloatSlider(value=1, min=0.25, max=8, step=0.25,
        description="Frequency (cycles/mm)", readout_format=".2f", continuous_update=False, style=style, layout=layout),
    sigma=widgets.FloatSlider(value=0.04, min=0, max=0.5, step=0.01,
        description="PSF width σ (mm)", readout_format=".2f", continuous_update=False, style=style, layout=layout),
    pitch=widgets.SelectionSlider(options=[(f"{p:.2f} mm", p) for p in PITCHES_MM], value=0.10,
        description="Pixel spacing p (mm)", continuous_update=False, style=style, layout=layout),
    phase_deg=widgets.FloatSlider(value=0, min=0, max=360, step=5,
        description="Target phase (degrees)", readout_format=".0f", continuous_update=False, style=style, layout=layout))
phase_panel = widgets.Accordion([controls["phase_deg"]], selected_index=None)
phase_panel.set_title(0, "Optional extension: target phase")
metrics_box = widgets.HTML()
plot_image = widgets.Image(format="png", layout=widgets.Layout(width="100%", max_width="1400px"))
updating_preset = False

def refresh(change=None):
    if updating_preset:
        return
    values = {key: control.value for key, control in controls.items()}
    metrics_box.value = metric_html(**values)
    import io
    fig = make_dashboard(**values)
    try:
        buffer = io.BytesIO()
        fig.savefig(buffer, format="png", dpi=110)
        plot_image.value = buffer.getvalue()
    finally:
        plt.close(fig)

PRESETS = {
    "Fine sampling": dict(f=1, sigma=0.04, pitch=0.10, phase_deg=0),
    "Blur limited": dict(f=1, sigma=0.25, pitch=0.10, phase_deg=0),
    "Aliasing": dict(f=3, sigma=0.02, pitch=0.25, phase_deg=0),
    "Nyquist boundary": dict(f=2, sigma=0.02, pitch=0.25, phase_deg=0)}

def set_preset(name):
    global updating_preset
    phase_panel.selected_index = 0 if name == "Nyquist boundary" else None
    updating_preset = True
    try:
        for key, value in PRESETS[name].items():
            controls[key].value = value
    finally:
        updating_preset = False
    refresh()

buttons = []
for name in PRESETS:
    button = widgets.Button(description=name, layout=widgets.Layout(width="150px"))
    button.on_click(lambda _, selected=name: set_preset(selected))
    buttons.append(button)
for control in controls.values():
    control.observe(refresh, names="value")
explorer = widgets.VBox([
    widgets.HTML("<b>Choose a starting point, then change one variable at a time.</b>"),
    widgets.Box(buttons, layout=widgets.Layout(display="flex", flex_flow="row wrap")),
    controls["f"], controls["sigma"], controls["pitch"], phase_panel, metrics_box, plot_image])
display(explorer)
refresh()


## 3 · Investigate and explain
Use the measurements as well as the images. Change **one variable at a time**. Record the settings and results in the table below. Write your answers in this notebook and submit your modified notebook file (`.ipynb`) to Canvas.

| Trial | $f$ (cycles/mm) | $\sigma$ (mm) | Pixel spacing $p$ (mm) | $f_N$ (cycles/mm) | Apparent $f$ (cycles/mm) | MTF at target | Observation |
|---|---|---|---|---|---|---|---|
| Fine sampling | | | | | | | |
| Blur limited | | | | | | | |
| Aliasing | | | | | | | |

**1 · Blur and contrast.** Compare **Fine sampling** with **Blur limited**. Calculate PSF FWHM for each using $\mathrm{FWHM}\approx2.355\sigma$. Record the MTF at 1 cycle/mm and the MTF50. Explain why the stripes lose contrast even though the detector has many pixels per stripe period.

*Answer:*

**2 · Pixel size and sampling.** Set $f=1$ cycle/mm, $\sigma=0$, and phase $=0°$. Compare pixel spacings $p=0.10$ and $p=0.40$ mm. Calculate the stripe period, the number of pixels per period, and $f_N$ for each. Both settings are below Nyquist. Use the pixel-averaging curve to explain why the larger pixels still reduce contrast.

*Answer:*

**3 · An apparent pattern.** Select **Aliasing**. Calculate $f_s=1/p$ and the apparent frequency $f_s-f$. Check your result against the profile and the reported apparent frequency. Change only pixel spacing to 0.10 mm. Explain why visible stripes in the coarse detector did not establish that the original stripe spacing was recorded correctly.

*Answer:*

**4 · Blur as a tradeoff.** Select **Aliasing**, then increase $\sigma$ from 0.02 to 0.15 mm at fixed frequency and pixel spacing. Record the predicted contrast after pixel averaging at both settings. Explain why blur reduces the false pattern and what detail it also removes. Would smoothing the displayed detector image recover the original stripe spacing? Explain.

*Answer:*

## Optional extension · Phase at the Nyquist frequency
Select **Nyquist boundary** and open **Optional extension: target phase**. At this setting $f=f_N$, there are exactly two samples per sinusoidal period. Compare phase 0° with 90°.

The exact pixel values in this model are

$$
d_n=\tfrac12[1+C_0H_{\rm opt}(f)\,\mathrm{sinc}(pf)\cos(2\pi f x_n+\phi)],
\qquad x_n=(n+\tfrac12)p.
$$

The phase slider is in degrees; use radians when substituting $\phi$ into the cosine. At $f=f_N$, the spatial term simplifies to $2\pi f x_n=\pi(n+\tfrac12)$.

**5 · Phase and the recorded pattern.** Describe how the detector image changes between phases 0° and 90°. Use the cosine above to explain the difference. Why does having exactly two samples per period fail to preserve arbitrary sinusoidal amplitude and phase?

Under **Additional measurements**, the sinusoidal fit estimates contrast from the samples. At zero apparent frequency or exactly Nyquist it reports *indeterminate*: there is not enough independent information to separate amplitude and phase. The **sample range contrast** instead uses the recorded maximum and minimum; these need not coincide with the continuous sinusoid's extrema. The dashed profile curve is a fit through the samples at the apparent frequency.

*Answer:*

## Model assumptions and further reading
The simulation uses exact expressions for the sinusoid after Gaussian blur and pixel averaging. The displayed 8 mm field is cropped from an infinite stripe pattern, so there are no object edges to blur. All pixels have the same sensitivity, and there are no random fluctuations in the recorded values. The model therefore isolates spatial resolution.

The Gaussian response has no sharp frequency cutoff. Blur can make an alias much weaker, but it does not strictly eliminate all frequencies above Nyquist. A negative sinc response shifts the stripes by half a period; MTF reports its magnitude. The simulator includes these effects even when they are not the focus of the required investigations.

- [Robertson (2017), *Detector sampling of optical/IR spectra: how many pixels per FWHM?*](https://arxiv.org/abs/1707.06455) discusses finite pixel size and sampling phase.

**If the controls stop responding:** reconnect the Colab runtime and choose **Runtime → Run all**. In JupyterLab, restart the kernel and run all cells. A GitHub preview is static; interaction requires a running notebook session.
